### Dependencies

In [2]:
'''
Visualization tools are adapted from https://github.com/facebookresearch/dino.
'''

# Base Dependencies
import argparse
import colorsys
import os
import random
import sys
import requests
from io import BytesIO

# LinAlg / Stats / Plotting Dependencies
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import skimage.io
from skimage.measure import find_contours
from tqdm import tqdm

# Torch Dependencies
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms as pth_transforms
import numpy as np
from PIL import Image

# Utils
import models.vision_transformer as vits
from attention_visualization_utils import create_256x256_map_concat

### Loading Pretrained ViT

In [3]:
arch = 'vit_tiny'
patch_size = 16
pretrained_weights = './ckpts/dino.pth'
checkpoint_key = 'teacher'
threshold = 0.5

device = torch.device("cpu")

### Build model
model = vits.__dict__[arch](patch_size=patch_size, num_classes=0)
for p in model.parameters():
    p.requires_grad = False
model.eval()
model.to(device)

if os.path.isfile(pretrained_weights):
    state_dict = torch.load(pretrained_weights, map_location="cpu")
    if checkpoint_key is not None and checkpoint_key in state_dict:
        print(f"Take key {checkpoint_key} in provided checkpoint dict")
        state_dict = state_dict[checkpoint_key]
    # remove `module.` prefix
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    # remove `backbone.` prefix induced by multicrop wrapper
    state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
    msg = model.load_state_dict(state_dict, strict=False)
    print('Pretrained weights found at {} and loaded with msg: {}'.format(pretrained_weights, msg))

In [ ]:
# Directory containing the images
input_dir = './image_demo/'
output_dir = './attention_visualization_results/image_256/'

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# List all files in the input directory
image_files = [f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, f)) and f.endswith('.png')]

# Loop over each image file
for img_fname in image_files:
    img_path = os.path.join(input_dir, img_fname)
    img = Image.open(img_path)
    create_256x256_map_concat(model, img, img_fname, output_dir, threshold=threshold, which_concat=[0, 1, 2], display=True)
    print(f"Processed {img_fname}")

# which_concat contains head index. vit_tiny has 3 heads
# if you want to visualize all of them, set which_concat to [0,1,2]